In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time
import joblib
import optuna
import optuna.visualization as vis

from pinn_core import (
    set_seed, pinn_architecture, sample_points, generate_noisy_data,
    compute_loss, train_forward, compute_loss_inverse, train_inverse,
    fd_solver, cn_nls_baseline,
)

torch.manual_seed(123)
np.random.seed(123)

#for traing_forward and train_inverse: defaults to GPU if available, else CPU.
default_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Auto-detected default device: {default_device}")

INVERSE START
**TODO:** Explain the inverse problem's baseline setup and how it mirrors the forward baseline's methodology.

In [ ]:
# Baseline (no-tuning) inverse PINN: the exact hyperparameters originally
# used in heat_pinn_basic.ipynb's inverse cell (hardcoded tanh activation,
# all loss terms unweighted), the same way forward_config captures the
# original forward hyperparameters.
baseline_inverse_config = {
    "activation":   "tanh",
    "hidden_size":  20,
    "n_layers":     4,
    "lambda_pde":   1.0,
    "lambda_bc":    1.0,
    "lambda_ic":    1.0,
    "lambda_data":  1.0,
    "N_f":          10000,
    "N_bc":         200,
    "N_ic":         200,
    "N_obs":        10,
    "noise_std":    0.05,
    "alpha_init":   0.1,
    "true_alpha":   0.4,
    "adam_lr":      1e-3,
    "adam_iters":   5000,
    "lbfgs_iters":  500,
}

In [ ]:
# Generate the baseline's own observations once, kept fixed across all 5
# seeds below -- same reasoning as the shared x_obs_inv/t_obs_inv/u_obs_inv
# used for Optuna trials further down: comparing seeds should only reflect
# training stochasticity, not a fresh random observation draw each time.
set_seed(0)
x_obs_baseline, t_obs_baseline, u_obs_baseline = generate_noisy_data(baseline_inverse_config)

baseline_inverse_results = []
for seed in range(5):
    set_seed(seed)
    r = train_inverse(baseline_inverse_config, x_obs_baseline, t_obs_baseline, u_obs_baseline, print_training=False)
    baseline_inverse_results.append(r)

baseline_alpha_errors = [r["alpha_error"] for r in baseline_inverse_results]
baseline_inverse_times = [r["train_time"] for r in baseline_inverse_results]

print(f"\nBaseline (no tuning) inverse PINN:")
print(f"  Alpha error: {np.mean(baseline_alpha_errors):.6e} ± {np.std(baseline_alpha_errors):.6e}")
print(f"  Time: {np.mean(baseline_inverse_times):.2f} ± {np.std(baseline_inverse_times):.2f} sec")

In [ ]:
# Parameters for generating the inverse problem's synthetic noisy
# observations -- named once here (rather than as a bare literal) since
# forthcoming sensitivity sweeps will vary N_obs and noise_std, and every
# generate_noisy_data(...) call in this section (search stage here, plus
# the confirmation-round and final-evaluation cells below) should draw
# from the same definition of the observation setup rather than each
# retyping its own copy of the same three numbers.
INVERSE_OBS_CONFIG = {
    "N_obs": 50,
    "noise_std": 0.05,
    "true_alpha": 0.4,
}

# Generate the inverse-problem observations once, shared across every Optuna
# trial, so trials are compared on the identical noisy dataset rather than
# each drawing its own random noise realization.
set_seed(0)
x_obs_inv, t_obs_inv, u_obs_inv = generate_noisy_data(INVERSE_OBS_CONFIG)

# Everything about the inverse problem that Optuna does NOT search over --
# defined once here so both objective_inverse (HPO trials) and the
# multi-seed retrain loop later (after be0839c3) build configs from the
# same source instead of two copies that could drift out of sync. N_obs
# and noise_std are deliberately not here: train_inverse never reads them
# (observations are passed in pre-built above), so including them would
# just be dead weight that looks like it configures something it doesn't.
#
# activation, N_f, and the N_bc/N_ic range now match forward's search space
# exactly (previously activation was fixed to "sin" and N_f fixed to 10000,
# with N_bc/N_ic searched over double forward's range) -- decided August 21,
# 2026: the original narrower inverse space was sized for MacBook CPU
# training time, not for any physics-based reason the two problems should
# search differently. Forward's own completed HPO run found tanh
# outperforming sin on a closely related problem, which undercuts the
# "sin is the right basis so fix it" assumption inverse was built on.
# Whether inverse specifically benefits from denser BC/IC point coverage
# (a real, separate hypothesis -- more boundary/IC anchoring could help
# alpha identifiability given only sparse noisy data) is deferred to a
# dedicated follow-up ablation rather than left as an untested asymmetry
# baked into the main comparison.
INVERSE_FIXED_CONFIG = {
    "lambda_pde":  1.0,
    "alpha_init":  0.1,
    "true_alpha":  0.4,
}

def objective_inverse(trial):
    config = {
        **INVERSE_FIXED_CONFIG,
        "hidden_size":  trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
        "n_layers":     trial.suggest_categorical("n_layers", [3, 4, 5]),
        "activation":   trial.suggest_categorical("activation", ["tanh", "sin"]),
        "lambda_bc":    trial.suggest_float("lambda_bc",  0.001, 1000.0, log=True),
        "lambda_ic":    trial.suggest_float("lambda_ic",  0.001, 1000.0, log=True),
        "lambda_data":  trial.suggest_float("lambda_data",  0.001, 1000.0, log=True),
        "N_f":          trial.suggest_categorical("N_f", [5000, 10000, 20000]),
        "N_bc":         trial.suggest_categorical("N_bc", [100, 200, 400]),
        "N_ic":         trial.suggest_categorical("N_ic", [100, 200, 400]),
        "adam_lr":      trial.suggest_float("adam_lr", 1e-4, 1e-2, log=True),
        "adam_iters":   1500,
        "lbfgs_iters":  500,
    }

    results = train_inverse(config, x_obs_inv, t_obs_inv, u_obs_inv, print_training=False, trial=trial)
    return abs(results["alpha"] - config["true_alpha"])

In [ ]:
if os.path.exists("optuna_inverse.db"):
    os.remove("optuna_inverse.db")


**TODO:** Note anything different from the forward search worth calling out (the extra lambda_data term, the search/confirmation/final-evaluation split) before the 300-trial run below.

In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="inverse_pinn_hpo",
    storage="sqlite:///optuna_inverse.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=800, interval_steps=200, n_min_trials=8),
    sampler=optuna.samplers.TPESampler(seed=42)
)

optuna_inverse_search_start = time.time()
study.optimize(
    objective_inverse,
    n_trials=300,
    show_progress_bar=True
)
optuna_inverse_search_time = time.time() - optuna_inverse_search_start

joblib.dump(study, 'inverse_hpo_study.pkl')

print(f"\nTotal Optuna inverse search time: {optuna_inverse_search_time:.2f}s")
print("\nBest trial:")
print(f"  alpha_error: {study.best_trial.value:.4e}")
print(f"  params: {study.best_trial.params}")

# Log-scale y-axis -- objective values span several orders of magnitude
# (early trials near 1, converged trials down near 1e-4), so a linear
# axis crowds nearly every trial against the best-value line and hides
# the actual spread. plot_optimization_history has no built-in log-scale
# option, so the returned plotly figure is updated directly.
opt_history_fig = vis.plot_optimization_history(study)
opt_history_fig.update_yaxes(type="log")
opt_history_fig.show()
vis.plot_param_importances(study).show()

In [ ]:
study = optuna.load_study(
    study_name="inverse_pinn_hpo",
    storage="sqlite:///optuna_inverse.db"
)

print(f"\nBest trial:")
print(f"  Alpha error: {study.best_value:.6e}")
print(f"  Config: {study.best_params}")

# See all top configs
print(f"\nTop 5 trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"{i}. Alpha error: {trial.value:.6e}")
    print(f"   Params: {trial.params}\n")

# Stats
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Summary:")
print(f"  Completed: {len(completed)}")
print(f"  Pruned: {len(pruned)}")
print(f"  Pruning saved: {len(pruned)/(len(completed)+len(pruned))*100:.1f}% of trials")


**TODO:** Explain the three-stage search/confirmation/final-evaluation split and why it exists (this is the leakage-prevention design worth walking through carefully), and note what the loss-curve/alpha-convergence plot below it is illustrating.

In [ ]:
# Confirmation round: retrain each of Optuna's top 3 configs (by search-
# stage alpha error) across 5 FRESH noisy datasets the search stage never
# saw (seeds 1-5, vs. the search stage's seed 0), and pick the winner by
# mean alpha error across those datasets. Since alpha only exists as a
# byproduct of training on some dataset -- there's no way to "query" a
# trained model against new data the way forward's field can be queried at
# new (x,t) points -- the inverse-problem equivalent of forward's held-out
# test grid is a held-out set of simulated noise realizations. This stage
# only decides WHICH CONFIG WINS; the final accuracy number is reported by
# the final-evaluation cell below on yet another, still-untouched set of
# datasets, so no single dataset both picks a winner and reports its
# accuracy.
top_inverse_configs = [
    {**INVERSE_FIXED_CONFIG, **trial.params, "adam_iters": 3000, "lbfgs_iters": 1500}
    for trial in sorted_trials[:3]
]

all_inverse_config_results = []

for i, config in enumerate(top_inverse_configs, 1):
    config_results = []
    for seed in range(1, 6):
        set_seed(seed)
        x_obs_c, t_obs_c, u_obs_c = generate_noisy_data(INVERSE_OBS_CONFIG)
        r = train_inverse(config, x_obs_c, t_obs_c, u_obs_c, print_training=False)
        config_results.append(r)

    errors = [r["alpha_error"] for r in config_results]
    times = [r["train_time"] for r in config_results]
    print(f"\nInverse config {i}:")
    print(f"  Alpha error: {np.mean(errors):.6e} ± {np.std(errors):.6e}")
    print(f"  Time:        {np.mean(times):.2f} ± {np.std(times):.2f} sec")

    all_inverse_config_results.append(config_results)

# Pick the winning config by its MEAN alpha error across the 5 confirmation
# datasets, not by any single individual run's error.
inverse_config_mean_errors = [np.mean([r["alpha_error"] for r in cr]) for cr in all_inverse_config_results]
best_inverse_config_idx = int(np.argmin(inverse_config_mean_errors))
winning_inverse_config = top_inverse_configs[best_inverse_config_idx]

print(f"\nWinning inverse config (by confirmation-round mean alpha error): Config {best_inverse_config_idx + 1}")

In [ ]:
# Final evaluation: retrain the confirmation round's winner -- frozen, no
# further config selection happens here -- across 5 MORE fresh datasets
# (seeds 6-10), never touched by the search stage or the confirmation
# round. This mean +/- std is what should be quoted as the tuned inverse
# PINN's accuracy, and CN-NLS in the next cell is evaluated on these
# identical datasets for a leakage-free final comparison.
final_inverse_results = []
for seed in range(6, 11):
    set_seed(seed)
    x_obs_f, t_obs_f, u_obs_f = generate_noisy_data(INVERSE_OBS_CONFIG)
    r = train_inverse(winning_inverse_config, x_obs_f, t_obs_f, u_obs_f, print_training=False)
    final_inverse_results.append(r)

tuned_inverse_alpha_error_mean = float(np.mean([r["alpha_error"] for r in final_inverse_results]))
tuned_inverse_alpha_error_std = float(np.std([r["alpha_error"] for r in final_inverse_results]))
tuned_inverse_train_time_mean = float(np.mean([r["train_time"] for r in final_inverse_results]))
tuned_inverse_train_time_std = float(np.std([r["train_time"] for r in final_inverse_results]))

print(f"\nFinal tuned inverse PINN (winning config, evaluated on fresh datasets):")
print(f"  Alpha error:   {tuned_inverse_alpha_error_mean:.6e} ± {tuned_inverse_alpha_error_std:.6e}")
print(f"  Training time: {tuned_inverse_train_time_mean:.2f} ± {tuned_inverse_train_time_std:.2f} sec")

# Use whichever seed's alpha error is closest to the mean as a single
# representative run for the illustrative plot below -- mirrors forward's
# representative-model selection in the top-configs cell.
representative_inverse_idx = int(np.argmin(
    [abs(r["alpha_error"] - tuned_inverse_alpha_error_mean) for r in final_inverse_results]))
inverse_results = final_inverse_results[representative_inverse_idx]

In [ ]:
# Illustrative loss-curve / alpha-convergence plot for the winning tuned
# config's representative run (from the retrain loop above) -- mirrors
# forward's representative-run plot, which shows the actual winning model
# rather than a separately hand-set config.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(inverse_results["history_total"], label='Total')
axes[0].semilogy(inverse_results["history_pde"], label='PDE')
axes[0].semilogy(inverse_results["history_bc"], label='BC')
axes[0].semilogy(inverse_results["history_ic"], label='IC')
axes[0].semilogy(inverse_results["history_data"], label='Data')
axes[0].axvline(x=inverse_results["adam_iters"], color='black', linestyle='--', label='Adam → L-BFGS')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Inverse PINN Training Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(inverse_results["history_alpha"], label='Recovered alpha')
axes[1].axhline(y=INVERSE_FIXED_CONFIG["true_alpha"], color='r', linestyle='--', label='True alpha')
axes[1].axvline(x=inverse_results["adam_iters"], color='black', linestyle='--', label='Adam → L-BFGS')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Alpha')
axes[1].set_title('Alpha Convergence')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

**TODO:** Explain what the CN-NLS comparison and the Baseline/Tuned/CN-NLS table below are comparing, and why CN-NLS has no training-time cost line.

In [ ]:
true_alpha = INVERSE_FIXED_CONFIG["true_alpha"]

# Run CN-NLS against the SAME 5 final-evaluation datasets as the tuned PINN
# above (each stored on its own result dict by train_inverse), so PINN and
# CN-NLS are compared on identical observations and CN-NLS gets the same
# across-datasets robustness treatment instead of being judged on a single
# noise realization.
cn_nls_results = []
for r in final_inverse_results:
    best_alpha, mse_history, alpha_candidates, cn_nls_time = cn_nls_baseline(
        {"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
    cn_nls_results.append({
        "best_alpha": best_alpha,
        "mse_history": mse_history,
        "alpha_candidates": alpha_candidates,
        "cn_nls_time": cn_nls_time,
        "alpha_error": abs(best_alpha - true_alpha),
    })

cn_nls_alpha_errors = [c["alpha_error"] for c in cn_nls_results]
cn_nls_times = [c["cn_nls_time"] for c in cn_nls_results]
cn_nls_alpha_error_mean = float(np.mean(cn_nls_alpha_errors))
cn_nls_alpha_error_std = float(np.std(cn_nls_alpha_errors))
cn_nls_time_mean = float(np.mean(cn_nls_times))
cn_nls_time_std = float(np.std(cn_nls_times))

# Use whichever draw's alpha error is closest to the mean as a single
# representative run for the landscape plot below -- illustrative only.
representative_cn_idx = int(np.argmin([abs(c["alpha_error"] - cn_nls_alpha_error_mean) for c in cn_nls_results]))
best_alpha = cn_nls_results[representative_cn_idx]["best_alpha"]
mse_history = cn_nls_results[representative_cn_idx]["mse_history"]
alpha_candidates = cn_nls_results[representative_cn_idx]["alpha_candidates"]
cn_nls_time = cn_nls_results[representative_cn_idx]["cn_nls_time"]
cn_nls_alpha_error = cn_nls_results[representative_cn_idx]["alpha_error"]

print(f"CN-NLS recovered alpha: {best_alpha:.6f} | True alpha: {true_alpha} | Error: {cn_nls_alpha_error:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(alpha_candidates, mse_history, label='MSE')
plt.axvline(x=best_alpha, color='b', linestyle='--', label=f'Best alpha ({best_alpha:.4f})')
plt.axvline(x=true_alpha, color='r', linestyle='--', label=f'True alpha ({true_alpha})')
plt.xlabel('Alpha candidate')
plt.ylabel('MSE')
plt.title('CN-NLS Objective Landscape (representative draw)')
plt.legend()
plt.grid(True)
plt.show()

# Baseline / Tuned / CN-NLS comparison table, mirroring forward's final
# summary table -- alpha error and cost, side by side. Row label says
# "runs" rather than "seeds": Baseline's 5 runs share one dataset (only
# training seed varies), but Tuned's 5 runs each use a different dataset,
# and CN-NLS has no training randomness at all -- its variation is entirely
# from the different datasets. "Seeds" would only be accurate for Baseline.
print(f"\n{'':25} {'Baseline':>14} {'Tuned':>14} {'CN-NLS':>14}")
print(f"{'-'*70}")
print(f"{'Alpha error':25} {'N/A':>14} {'N/A':>14} {cn_nls_alpha_error:>14.4e}")
print(f"{'  mean ± std (5 runs)':25} {f'{np.mean(baseline_alpha_errors):.4e}±{np.std(baseline_alpha_errors):.4e}':>14} {f'{tuned_inverse_alpha_error_mean:.4e}±{tuned_inverse_alpha_error_std:.4e}':>14} {f'{cn_nls_alpha_error_mean:.4e}±{cn_nls_alpha_error_std:.4e}':>14}")
print(f"{'Training/opt time (s)':25} {'N/A':>14} {'N/A':>14} {cn_nls_time:>14.2f}")
print(f"{'  mean ± std (5 runs)':25} {f'{np.mean(baseline_inverse_times):.2f}±{np.std(baseline_inverse_times):.2f}':>14} {f'{tuned_inverse_train_time_mean:.2f}±{tuned_inverse_train_time_std:.2f}':>14} {f'{cn_nls_time_mean:.2f}±{cn_nls_time_std:.2f}':>14}")
print(f"{'Optuna search time (s)':25} {'N/A':>14} {optuna_inverse_search_time:>14.2f} {'N/A':>14}")

**TODO:** (After the final run) Summarize how the tuned PINN compares to CN-NLS on accuracy and cost, and state your position on the paper's framing for this result.

In [ ]:
# CPU-only control-timing run: same winning frozen inverse config as the
# GPU numbers reported above, explicit device="cpu", reusing the exact same
# 5 final-evaluation observation datasets (seeds 6-10) already generated for
# the GPU run -- so this is a hardware-context data point, not a repeat of
# the leakage-checked accuracy claim on different data.
cpu_inverse_results = []
for seed, r_gpu in zip(range(6, 11), final_inverse_results):
    set_seed(seed)
    r = train_inverse(winning_inverse_config, r_gpu["x_obs"], r_gpu["t_obs"], r_gpu["u_obs"],
                       print_training=False, device=torch.device("cpu"))
    cpu_inverse_results.append(r)

cpu_inverse_train_time_mean = float(np.mean([r["train_time"] for r in cpu_inverse_results]))
cpu_inverse_train_time_std = float(np.std([r["train_time"] for r in cpu_inverse_results]))

print(f"\nCPU control (winning inverse config, device='cpu', 5 runs):")
print(f"  Training time: {cpu_inverse_train_time_mean:.2f} ± {cpu_inverse_train_time_std:.2f} sec  (GPU: {tuned_inverse_train_time_mean:.2f} ± {tuned_inverse_train_time_std:.2f} sec)")

**TODO:** Explain why these three sweeps (noise, observation count, true_alpha) exist and what each one tests about the tuned config's robustness.

In [ ]:
# Sensitivity sweep 1: vary observation noise (noise_std) at fixed
# N_obs=50, comparing the tuned PINN (winning_inverse_config, frozen from
# the confirmation round above -- NOT re-tuned per point, since the point
# is testing how robust the already-selected config is to noise, not
# re-running the expensive HPO search 4 more times) against CN-NLS at each
# noise level. The noise_std=0.05 point reuses the already-computed final
# results above instead of retraining a duplicate of the identical
# experiment. Seeds 11-15 used here -- search=0, confirmation=1-5,
# final-eval=6-10, this sweep=11-15, the N_obs sweep below=16-20 -- so no
# dataset is reused across any two stages, including between the sweeps.
NOISE_SWEEP_VALUES = [0.01, 0.05, 0.1, 0.2]
noise_sweep_results = {}
noise_sweep_start = time.time()

for noise_std in NOISE_SWEEP_VALUES:
    if noise_std == INVERSE_OBS_CONFIG["noise_std"]:
        noise_sweep_results[noise_std] = {
            "pinn_alpha_error_mean": tuned_inverse_alpha_error_mean,
            "pinn_alpha_error_std": tuned_inverse_alpha_error_std,
            "cn_nls_alpha_error_mean": cn_nls_alpha_error_mean,
            "cn_nls_alpha_error_std": cn_nls_alpha_error_std,
        }
        print(f"noise_std={noise_std}: reused final-evaluation results (identical config)")
        continue

    sweep_obs_config = {**INVERSE_OBS_CONFIG, "noise_std": noise_std}
    pinn_errors, cn_errors = [], []
    for seed in range(11, 16):
        set_seed(seed)
        x_obs_s, t_obs_s, u_obs_s = generate_noisy_data(sweep_obs_config)
        r = train_inverse(winning_inverse_config, x_obs_s, t_obs_s, u_obs_s, print_training=False)
        pinn_errors.append(r["alpha_error"])

        best_alpha_s, _, _, _ = cn_nls_baseline({"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
        cn_errors.append(abs(best_alpha_s - true_alpha))

    noise_sweep_results[noise_std] = {
        "pinn_alpha_error_mean": float(np.mean(pinn_errors)),
        "pinn_alpha_error_std": float(np.std(pinn_errors)),
        "cn_nls_alpha_error_mean": float(np.mean(cn_errors)),
        "cn_nls_alpha_error_std": float(np.std(cn_errors)),
    }
    print(f"noise_std={noise_std}: PINN alpha error {np.mean(pinn_errors):.4e} ± {np.std(pinn_errors):.4e} | "
          f"CN-NLS alpha error {np.mean(cn_errors):.4e} ± {np.std(cn_errors):.4e}")

noise_sweep_time = time.time() - noise_sweep_start
print(f"\nNoise sensitivity sweep total time: {noise_sweep_time:.2f}s")

plt.figure(figsize=(7, 5))
xs = sorted(noise_sweep_results.keys())
pinn_means = [noise_sweep_results[x]["pinn_alpha_error_mean"] for x in xs]
pinn_stds = [noise_sweep_results[x]["pinn_alpha_error_std"] for x in xs]
cn_means = [noise_sweep_results[x]["cn_nls_alpha_error_mean"] for x in xs]
cn_stds = [noise_sweep_results[x]["cn_nls_alpha_error_std"] for x in xs]
plt.errorbar(xs, pinn_means, yerr=pinn_stds, marker='o', label='Tuned PINN', capsize=4)
plt.errorbar(xs, cn_means, yerr=cn_stds, marker='s', label='CN-NLS', capsize=4)
plt.xlabel('Observation noise (noise_std)')
plt.ylabel('Alpha error')
plt.title(f'Alpha Error vs. Observation Noise (N_obs={INVERSE_OBS_CONFIG["N_obs"]})')
plt.legend()
plt.grid(True)
plt.show()

**TODO:** (After the final run) Summarize how accuracy degrades, or doesn't, as observation noise increases, for the PINN versus CN-NLS.

In [ ]:
# Sensitivity sweep 2: vary observation count (N_obs) at fixed
# noise_std=0.05, same methodology as the noise sweep above -- frozen
# winning_inverse_config, N_obs=50 point reuses the already-computed final
# results, seeds 16-20 (distinct from every other stage including the
# noise sweep's 11-15).
NOBS_SWEEP_VALUES = [10, 25, 50, 100]
nobs_sweep_results = {}
nobs_sweep_start = time.time()

for n_obs in NOBS_SWEEP_VALUES:
    if n_obs == INVERSE_OBS_CONFIG["N_obs"]:
        nobs_sweep_results[n_obs] = {
            "pinn_alpha_error_mean": tuned_inverse_alpha_error_mean,
            "pinn_alpha_error_std": tuned_inverse_alpha_error_std,
            "cn_nls_alpha_error_mean": cn_nls_alpha_error_mean,
            "cn_nls_alpha_error_std": cn_nls_alpha_error_std,
        }
        print(f"N_obs={n_obs}: reused final-evaluation results (identical config)")
        continue

    sweep_obs_config = {**INVERSE_OBS_CONFIG, "N_obs": n_obs}
    pinn_errors, cn_errors = [], []
    for seed in range(16, 21):
        set_seed(seed)
        x_obs_s, t_obs_s, u_obs_s = generate_noisy_data(sweep_obs_config)
        r = train_inverse(winning_inverse_config, x_obs_s, t_obs_s, u_obs_s, print_training=False)
        pinn_errors.append(r["alpha_error"])

        best_alpha_s, _, _, _ = cn_nls_baseline({"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
        cn_errors.append(abs(best_alpha_s - true_alpha))

    nobs_sweep_results[n_obs] = {
        "pinn_alpha_error_mean": float(np.mean(pinn_errors)),
        "pinn_alpha_error_std": float(np.std(pinn_errors)),
        "cn_nls_alpha_error_mean": float(np.mean(cn_errors)),
        "cn_nls_alpha_error_std": float(np.std(cn_errors)),
    }
    print(f"N_obs={n_obs}: PINN alpha error {np.mean(pinn_errors):.4e} ± {np.std(pinn_errors):.4e} | "
          f"CN-NLS alpha error {np.mean(cn_errors):.4e} ± {np.std(cn_errors):.4e}")

nobs_sweep_time = time.time() - nobs_sweep_start
print(f"\nN_obs sensitivity sweep total time: {nobs_sweep_time:.2f}s")

plt.figure(figsize=(7, 5))
xs = sorted(nobs_sweep_results.keys())
pinn_means = [nobs_sweep_results[x]["pinn_alpha_error_mean"] for x in xs]
pinn_stds = [nobs_sweep_results[x]["pinn_alpha_error_std"] for x in xs]
cn_means = [nobs_sweep_results[x]["cn_nls_alpha_error_mean"] for x in xs]
cn_stds = [nobs_sweep_results[x]["cn_nls_alpha_error_std"] for x in xs]
plt.errorbar(xs, pinn_means, yerr=pinn_stds, marker='o', label='Tuned PINN', capsize=4)
plt.errorbar(xs, cn_means, yerr=cn_stds, marker='s', label='CN-NLS', capsize=4)
plt.xlabel('Number of observations (N_obs)')
plt.ylabel('Alpha error')
plt.title(f'Alpha Error vs. Observation Count (noise_std={INVERSE_OBS_CONFIG["noise_std"]})')
plt.legend()
plt.grid(True)
plt.show()

**TODO:** (After the final run) Summarize how accuracy changes with fewer or more observations, for the PINN versus CN-NLS.

In [ ]:
# Sensitivity sweep 3: vary the inverse problem's true diffusivity
# (true_alpha) across the same values used in forward's alpha sweep above,
# so the two problems' generalization-beyond-alpha=0.4 results are directly
# comparable. Freezes winning_inverse_config -- only true_alpha changes per
# point, alpha_init stays fixed at its already-tuned starting value (0.1)
# rather than scaling with the swept alpha. This is deliberate, not an
# oversight: a real deployment of this same tuned recipe on a genuinely
# different diffusivity would not know to adjust alpha_init either, so
# keeping it fixed is the realistic test of how the recipe generalizes,
# not an unfair handicap. All sweep values must stay within CN-NLS's fixed
# [0.01, 1.0] search range (cn_nls_baseline's own alpha_candidates grid,
# unchanged per point) for CN-NLS to have any chance of finding them.
# The true_alpha=0.4 point reuses the already-computed final-evaluation/
# CN-NLS results. Seeds 21-25 used here, distinct from every other stage.
ALPHA_SWEEP_VALUES_INV = [0.1, 0.4, 0.7, 1.0]
inverse_alpha_sweep_results = {}
inverse_alpha_sweep_start = time.time()

for alpha_val in ALPHA_SWEEP_VALUES_INV:
    if alpha_val == INVERSE_OBS_CONFIG["true_alpha"]:
        inverse_alpha_sweep_results[alpha_val] = {
            "pinn_alpha_error_mean": tuned_inverse_alpha_error_mean,
            "pinn_alpha_error_std": tuned_inverse_alpha_error_std,
            "cn_nls_alpha_error_mean": cn_nls_alpha_error_mean,
            "cn_nls_alpha_error_std": cn_nls_alpha_error_std,
        }
        print(f"true_alpha={alpha_val}: reused final-evaluation results (identical config)")
        continue

    sweep_obs_config_a = {**INVERSE_OBS_CONFIG, "true_alpha": alpha_val}
    sweep_inverse_config_a = {**winning_inverse_config, "true_alpha": alpha_val}
    pinn_errors, cn_errors = [], []
    for seed in range(21, 26):
        set_seed(seed)
        x_obs_a, t_obs_a, u_obs_a = generate_noisy_data(sweep_obs_config_a)
        r = train_inverse(sweep_inverse_config_a, x_obs_a, t_obs_a, u_obs_a, print_training=False)
        pinn_errors.append(r["alpha_error"])

        best_alpha_a, _, _, _ = cn_nls_baseline({"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
        cn_errors.append(abs(best_alpha_a - alpha_val))

    inverse_alpha_sweep_results[alpha_val] = {
        "pinn_alpha_error_mean": float(np.mean(pinn_errors)),
        "pinn_alpha_error_std": float(np.std(pinn_errors)),
        "cn_nls_alpha_error_mean": float(np.mean(cn_errors)),
        "cn_nls_alpha_error_std": float(np.std(cn_errors)),
    }
    print(f"true_alpha={alpha_val}: PINN alpha error {np.mean(pinn_errors):.4e} ± {np.std(pinn_errors):.4e} | "
          f"CN-NLS alpha error {np.mean(cn_errors):.4e} ± {np.std(cn_errors):.4e}")

inverse_alpha_sweep_time = time.time() - inverse_alpha_sweep_start
print(f"\nInverse alpha sensitivity sweep total time: {inverse_alpha_sweep_time:.2f}s")

plt.figure(figsize=(7, 5))
xs = sorted(inverse_alpha_sweep_results.keys())
pinn_means = [inverse_alpha_sweep_results[x]["pinn_alpha_error_mean"] for x in xs]
pinn_stds = [inverse_alpha_sweep_results[x]["pinn_alpha_error_std"] for x in xs]
cn_means = [inverse_alpha_sweep_results[x]["cn_nls_alpha_error_mean"] for x in xs]
cn_stds = [inverse_alpha_sweep_results[x]["cn_nls_alpha_error_std"] for x in xs]
plt.errorbar(xs, pinn_means, yerr=pinn_stds, marker='o', label='Tuned PINN', capsize=4)
plt.errorbar(xs, cn_means, yerr=cn_stds, marker='s', label='CN-NLS', capsize=4)
plt.xlabel('True diffusivity (alpha)')
plt.ylabel('Alpha error')
plt.title('Alpha Error vs. True Diffusivity')
plt.legend()
plt.grid(True)
plt.show()

**TODO:** (After the final run) Summarize whether CN-NLS's advantage over the tuned PINN holds across all tested true_alpha values.